In [1]:
import simpulse 
import matplotlib.pyplot as plt
import numpy as np
import chime_frb_constants as const
from fitburst.routines.manipulate import downsample_2d
from syn_frbs import CreateSyntheticFRBs
import create_synthetic_parameters
from pathlib import Path
# Set the default font family
plt.rcParams['font.family'] = 'sans-serif'
import os
import random
import csv
import shutil

In [2]:
create_syn_frbs = CreateSyntheticFRBs(dm=0)

## Create many files for analyses

In [ ]:
def simple_peak_snr_auto(ds, frac=0.3):
    ts = ds.mean(axis=0)
    n = ts.size
    m = int(frac * n)
    off = np.r_[0:m, n-m:n]
    mu = ts[off].mean()
    sigma = ts[off].std(ddof=1)
    snr = (ts.max() - mu) / sigma
    return snr

In [ ]:
def append_to_csv(filepath, parameters):
    # Check if the fiel path already exists
    file_exists = Path(filepath).exists()
    with open(filepath, 'a', newline='') as csvfile:
        fieldnames = list(parameters.keys())
        if not file_exists:
            Path(filepath).touch()
            writer = csv.writer(csvfile)
            writer.writerow(fieldnames)
        writer = csv.writer(csvfile)
        writer.writerow(list(parameters.values()))   

j = 1
try:
    tar_path = "../scattering_timescale_dec_2/syn_data/fitburst_vs_mtgmdn/parameter.csv"
    os.remove(tar_path)
    for item in Path(tar_path).parent.glob("*.npy"):
        os.remove(item)
except Exception:
    print("No older files found to delete....")

for i in np.arange(1, 5, 0.1):   # RMS Values
    noise_rms = i
    scat_time_ms = 10
    intrinsic_width = 0.001   # seconds
    fluence = 0.005   # Original was 0.005
    central_freq = 500
    spectral_width = 80
    syn_frbs, noiseless, snr, noiseless_full, data_full = create_syn_frbs.gaussian_spectrum(spectral_index=0,
                                                                 fluence_jy_s=fluence,
                                                                 intrinsic_width_s=intrinsic_width,
                                                                 noise_rms=noise_rms,
                                                                 scat_time_ms=scat_time_ms,
                                                                 central_freq=central_freq,
                                                                 width=spectral_width,
                                                                 undispersed_arrival_time_factor=2  # 2 for center in time
                                                                                           )


    
    
    snr_calculated = simple_peak_snr_auto(data_full)
    
    parameters = {
            "noise_rms": noise_rms,
            "scat_time_ms": scat_time_ms,
            "intrinsic_width": intrinsic_width,
            "fluence": fluence,
            "central_freq": central_freq,
            "spectral_width": spectral_width,
            "snr": snr_calculated
        }

    
    append_to_csv(filepath=tar_path, parameters=parameters)

    np.save(f"../scattering_timescale_dec_2/syn_data/fitburst_vs_mtgmdn/scatter_{j}.npy", syn_frbs)
    j+=1

## Create Single File for analysis

In [ ]:
noise_rms = 4
scat_time_ms = 2
intrinsic_width = 0.001
fluence = 0.005   # Origincal was 0.005
central_freq = 600
spectral_width = 80
syn_frbs, noiseless, snr, noiseless_full, data_full = create_syn_frbs.gaussian_spectrum(spectral_index=0,
                                                             fluence_jy_s=fluence,
                                                             intrinsic_width_s=intrinsic_width,
                                                             noise_rms=noise_rms,
                                                             scat_time_ms=scat_time_ms,
                                                             central_freq=central_freq,
                                                             width=spectral_width,
                                                             undispersed_arrival_time_factor=2  # 2 for center in time
                                                                                       )



In [ ]:
data_full.shape

In [ ]:
plt.imshow(noiseless, aspect='auto', origin='lower')
snr

In [ ]:
plt.imshow(syn_frbs, aspect='auto', origin='lower')
snr

In [ ]:
fig, ax = plt.subplots(2, 1, height_ratios=[1, 3], figsize=(4, 4))
fig.subplots_adjust(hspace=0.03)
ax[0].plot(syn_frbs.mean(axis=0), color='black')
ax[0].set_facecolor("#ddddff")
ax[0].set_xticks([])
ax[0].set_yticks([])
ax[0].set_xlim(0, len(syn_frbs.mean(axis=0)))
for spine in ax[0].spines.values():
    spine.set_edgecolor("black")
    spine.set_linewidth(1)
ax[1].imshow(syn_frbs, aspect='auto', origin='lower')
ax[1].set_xlabel("Time [ms]")
ax[1].set_ylabel("Frequency [MHz]")
for spine in ax[1].spines.values():
    spine.set_edgecolor("black")
    spine.set_linewidth(1)
ax[1].set_yticks(np.linspace(0, 256, 9), np.linspace(400, 800, 9).astype(int))
ax[1].minorticks_on()
plt.savefig("../scatter_example.jpg", bbox_inches='tight', dpi=600)
plt.show()

## Create a npz file for the fitburst

In [ ]:
metadata = {
    "bad_chans"      : [], # a Python list of indices corresponding to frequency channels to zero-weight
    "freqs_bin0"     : const.FREQ_BOTTOM_MHZ, # a floating-point scalar indicating the value of frequency bin at index 0, in MHz
    "is_dedispersed" : True, # a boolean indicating if spectrum is already dedispersed (True) or not (False)
    "num_freq"       : const.NUM_CHANNELS,  # an integer scalar indicating the number of frequency bins/channels
    "num_time"       : 162,# an integer scalar indicating the number of time bins
    "times_bin0"     : 1,# a floating-point scalar indicating the value of time bin at index 0, in MJD
    "res_freq"       : const.CHANNEL_BANDWIDTH_MHZ,# a floating-point scalar indicating the frequency resolution, in MHz
    "res_time"       : const.SAMPLING_TIME_S# a floating-point scalar indicating the time resolution, in seconds
}
burst_parameters = {
    "amplitude"            :[0.5], # a list containing the the log (base 10) of the overall signal amplitude
    "arrival_time"         :[0.07], # a list containing the arrival times, in seconds
    "burst_width"          :[0.001],# a list containing the temporal widths, in seconds
    "dm"                   :[0],# a list containing the dispersion measures (DM), in parsec per cubic centimeter
    "dm_index"             :[-2],# a list containing the exponents of frequency dependence in DM delay
    "ref_freq"             :[400.1953125],# a list containing the reference frequencies for arrival-time and power-law parameter estimates, in MHz (held fixed)
    "scattering_index"     :[-4],# a list containing the exponents of frequency dependence in scatter-broadening
    "scattering_timescale" :[0.01], # a list containing the scattering timescales, in seconds
    "spectral_index"       :[0], # a list containing the power-law spectral indices
    "spectral_running"     :[-4] # a list containing the power-law spectral running
}
np.savez(
    "../scattering_timescale_dec_2/syn_data/scat_time.npz", 
    data_full=data_full, 
    metadata=metadata, 
    burst_parameters=burst_parameters
)

In [ ]:
# Save files to CSV
data = {
    "noise_rms": noise_rms,
    "scat_time_ms": scat_time_ms,
    "intrinsic_width": intrinsic_width,
    "fluence": fluence,
    "central_freq": central_freq,
    "spectral_width": spectral_width,
    "snr": snr
}

file_name = "../scattering_timescale_dec_2/syn_data/fluence_variations.csv"

# Check if file exists to decide if we need to write the header
file_exists = os.path.isfile(file_name)

# Open in 'a' (append) mode. 'newline=""' prevents blank lines on Windows.
with open(file_name, mode='a', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=data.keys())

    # Only write the header if the file is being created for the first time
    if not file_exists:
        writer.writeheader()
    
    # Append the data row
    writer.writerow(data)

print(f"Data successfully saved to {file_name}")